# 23N Field Tests #06 · Tomorrow's sun at Khavda

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft06-khavda-solar-forecast/ft06-khavda-solar-forecast/ft06_khavda_solar_forecast.ipynb)

**Question.** Can a small open time-series model forecast tomorrow's solar output at the Khavda renewable energy park
in Kutch better than "same as yesterday", and what does the difference save in grid deviation charges for 1 GW?

**Why it matters.** Under the CERC Deviation Settlement Mechanism (DSM) Regulations 2024, a solar seller is paid
for what it schedules and settles every 15-minute block in which actual output misses the schedule. From
1 April 2026 the free band for solar shrank from ±10 % to ±5 % of available capacity. Better forecasts mean
smaller misses.

**Method.** Five years of hourly ERA5 reanalysis sunlight and temperature for the park (via Open-Meteo) become the
output of a notional 1 GW AC plant through a simple, stated PV model. For every day of the test year three
forecasters predict the next 24 hours from data up to midnight: "same as yesterday" (persistence), a 7-day
average profile, and `amazon/chronos-bolt-small`, a 48M-parameter pretrained time-series model used zero-shot.
Each forecast becomes the schedule, and the deviation rule turns the misses into rupees.

**Data and licences.** ERA5 hourly reanalysis (Copernicus Climate Change Service) served by the Open-Meteo
Historical Weather API: **CC BY 4.0**, no key, no login. NASA POWER daily irradiance is used only as a
cross-check (NASA open data, no key). Model: `amazon/chronos-bolt-small`, **Apache 2.0**, not gated.

**Runtime.** About 3–6 minutes on free Colab (CPU is fine; a T4 is faster). No paid API keys.

*Independent study on public data by 23N AI Labs. "Khavda" is used only as a location: this models no real plant
and implies no affiliation with any developer, operator, utility, regulator or government body. All money figures
are estimates under the named assumptions below.*

## 1 · Setup

In [ ]:
# CELL: deps
# Colab and Kaggle already have torch and transformers. chronos-forecasting adds the Chronos-Bolt pipeline.
import os, sys, subprocess
if os.environ.get("FT06_SMOKE_TEST", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting", "gradio"], check=False)

In [ ]:
# CELL: config
import os, sys, io, json, math, time, textwrap, subprocess, warnings, urllib.request, urllib.parse, datetime as dt
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet: it replaces the weather download with a
# synthetic clear-sky-plus-clouds series and Chronos with a simple stand-in. Never post its output.
SMOKE_TEST = os.environ.get("FT06_SMOKE_TEST", "0") == "1"

FT_NUM = "06"
# ---- Site: Khavda renewable energy park (Gujarat Hybrid Renewable Energy Park), Kutch. Coordinates from the
# park's Wikipedia infobox. ERA5 cells are ~0.25°, so a few km either way changes little.
SITE_NAME = "Khavda RE park, Kutch"
LAT, LON = 24.117, 69.350
DATA_START, DATA_END = "2021-01-01", "2025-12-31"   # five full calendar years of hourly data
TEST_YEAR = 2025          # headline year: one forecast per day
ROBUST_YEAR = 2024        # robustness variant: the same test on the year before
TZ = "Asia/Kolkata"       # IST, no daylight saving

# ---- Notional plant (assumptions, printed with the result)
CAPACITY_MW = 1000        # AC capacity = available capacity in every block (no outages assumed)
DC_AC_RATIO = 1.25        # DC modules per unit of inverter capacity; output clips at CAPACITY_MW
PERF_RATIO = 0.80         # wiring, soiling, inverter and mismatch losses, excluding temperature
TEMP_COEFF = -0.0035      # per °C above 25 °C cell temperature (typical crystalline silicon)
NOCT_C = 45               # nominal operating cell temperature, for the cell-temperature estimate

# ---- Money and the deviation rule (CERC DSM Regulations 2024, Reg 8(4), solar / hybrid sellers)
CONTRACT_RATE_RS_PER_KWH = 2.60   # assumption: lowest bid in NTPC's 1.5 GW ISTS solar auction, Tranche I (Jan 2024)
BAND1_PCT = 5             # VLwS(1): deviation up to 5 % of the denominator is settled at the contract rate (no loss)
BAND2_PCT = 10            # VLwS(2): 5-10 % costs 10 % of the contract rate (paid 110 % / received 90 %)
BAND2_LOSS = 0.10         # loss per kWh in band 2, as a share of the contract rate
BEYOND_LOSS = 1.00        # beyond 10 %: under-injection paid at 200 %, over-injection received at 0 % → loss = 100 %
X_PCT = 100               # denominator = X % of available capacity + (100 - X) % of schedule; X = 100 in FY 2026-27
OLD_BAND1_PCT, OLD_BAND2_PCT = 10, 15   # robustness: the bands that applied until 31 March 2026
X_NEXT_PCT = 90           # robustness: X for solar in FY 2027-28 (CERC order on the value of X, 2026)
BLOCKS_PER_HOUR = 4       # DSM settles 15-minute blocks; hourly data → four identical blocks per hour

# ---- Forecasting
MODEL_ID = "amazon/chronos-bolt-small"   # 48M parameters, Apache 2.0
MODEL_LABEL, MODEL_PARAMS = "Chronos-Bolt-Small", "48M"
CONTEXT_DAYS = 28         # history handed to the model for each forecast
HORIZON_H = 24
BATCH = 64
CRORE = 1e7

# ---- Sanity-check bounds (what is already known about this site and about forecasting)
GHI_DAILY_RANGE = (4.8, 6.8)      # annual mean kWh/m²/day; Kutch is among India's sunniest regions (~5.5-6)
CF_RANGE = (0.15, 0.30)           # AC capacity factor of a fixed-tilt-like plant in Gujarat
PEAK_HOUR_RANGE = (12, 14)        # IST label of the sunniest hour (values are preceding-hour means)
MAX_GHI = 1250                    # W/m², above this the units are wrong
YEAR_BREAK_PCT = 8                # a year whose mean GHI differs from the median year by more → data-version break
NASA_RATIO_RANGE = (0.85, 1.15)   # Open-Meteo ERA5 vs NASA POWER annual mean GHI
PERSIST_NMAE_RANGE = (2, 30)      # % of capacity, daytime; persistence outside this means a bug
MAX_CREDIBLE_CUT = 0.70           # a > 70 % cut in charges from a zero-shot model would be suspicious (leakage?)

# ---- Where things live (this branch until it is merged)
REPO_REF = "ft06-khavda-solar-forecast"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"

# ---- Brand kit
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
DATA_CREDIT = "Data: ERA5 via Open-Meteo.com (CC BY 4.0)"
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic weather and a stand-in forecaster. These results are NOT real. Do NOT post.")

## 2 · Data: five years of hourly sunlight at the park

The Open-Meteo Historical Weather API serves ERA5 reanalysis for any point: `shortwave_radiation` is global
horizontal irradiance (GHI, W/m², mean over the preceding hour) and `temperature_2m` is air temperature (°C). We
ask for `models=era5` so all five years come from one consistent product (a mix of products is a data-version
break, which cost FT03 a whole run). Times are IST.

In [ ]:
# CELL: load_weather
def http_json(url, tries=4):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "23N-Field-Tests/1.0 (+https://23nlabs.com)"})
            with urllib.request.urlopen(req, timeout=120) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            last = e
            if e.code == 400:
                raise
        except Exception as e:
            last = e
        time.sleep(3 * (k + 1))
    raise last

def load_open_meteo(lat, lon, start, end, model="era5"):
    q = dict(latitude=lat, longitude=lon, start_date=start, end_date=end,
             hourly="shortwave_radiation,temperature_2m", timezone=TZ)
    if model:
        q["models"] = model
    url = "https://archive-api.open-meteo.com/v1/archive?" + urllib.parse.urlencode(q)
    js = http_json(url)
    h = js["hourly"]
    df = pd.DataFrame({"ghi": h["shortwave_radiation"], "t2m": h["temperature_2m"]},
                      index=pd.to_datetime(h["time"]))
    meta = dict(url=url, units=js.get("hourly_units", {}), utc_offset_s=js.get("utc_offset_seconds"),
                grid_lat=js.get("latitude"), grid_lon=js.get("longitude"), elevation=js.get("elevation"), model=model)
    return df, meta

def synthetic_weather(start, end, seed=6):
    # SMOKE TEST ONLY: clear-sky sun geometry for the site times a random cloud factor. Not real data.
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, pd.Timestamp(end) + pd.Timedelta(hours=23), freq="h")
    mid = idx - pd.Timedelta(minutes=30)                       # preceding-hour mean → sun position at mid-hour
    doy = mid.dayofyear.values; hr = mid.hour.values + mid.minute.values / 60
    decl = np.radians(23.45) * np.sin(2 * np.pi * (284 + doy) / 365)
    solar_time = hr + (LON - 82.5) / 15                          # IST meridian is 82.5° E
    ha = np.radians(15 * (solar_time - 12))
    lat = np.radians(LAT)
    sin_el = np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(ha)
    cs = np.where(sin_el > 0, 1098 * sin_el * np.exp(-0.057 / np.maximum(sin_el, 1e-3)), 0.0)
    n_days = len(idx) // 24
    months = idx[::24].month.values
    monsoon = np.isin(months, [6, 7, 8, 9])
    daily = np.clip(np.where(monsoon, 0.62, 0.9) + rng.normal(0, np.where(monsoon, 0.22, 0.08), n_days), 0.05, 1.0)
    for i in range(1, n_days):                                    # some day-to-day persistence in cloudiness
        daily[i] = 0.5 * daily[i] + 0.5 * daily[i - 1]
    cloud = np.repeat(daily, 24) * np.clip(1 + rng.normal(0, 0.06, len(idx)), 0.6, 1.1)
    ghi = np.clip(cs * cloud, 0, None)
    t2m = 27 + 6 * np.sin(2 * np.pi * (doy - 100) / 365) + 5 * np.sin(2 * np.pi * (hr - 9) / 24) + rng.normal(0, 1, len(idx))
    return pd.DataFrame({"ghi": ghi, "t2m": t2m}, index=idx), dict(url="synthetic (smoke test)", units={}, model="synthetic")

if SMOKE_TEST:
    WX, WX_META = synthetic_weather(DATA_START, DATA_END)
else:
    try:
        WX, WX_META = load_open_meteo(LAT, LON, DATA_START, DATA_END, "era5")
    except urllib.error.HTTPError as e:
        print("models=era5 was refused (", e, "); retrying with Open-Meteo's default model mix. The data-version check below will tell if that is a problem.")
        WX, WX_META = load_open_meteo(LAT, LON, DATA_START, DATA_END, None)

WX = WX.sort_index()
full_idx = pd.date_range(DATA_START, pd.Timestamp(DATA_END) + pd.Timedelta(hours=23), freq="h")
N_MISSING = int(WX.reindex(full_idx)["ghi"].isna().sum())
WX = WX.reindex(full_idx)
WX["ghi"] = WX["ghi"].interpolate(limit=3).fillna(0.0).clip(lower=0)
WX["t2m"] = WX["t2m"].interpolate(limit_direction="both")
print(f"{len(WX):,} hourly rows {WX.index[0]} → {WX.index[-1]} | missing hours filled: {N_MISSING}")
print("source:", WX_META["url"][:160])
print("units:", WX_META.get("units"), "| UTC offset (s):", WX_META.get("utc_offset_s"),
      "| grid cell:", WX_META.get("grid_lat"), WX_META.get("grid_lon"))

In [ ]:
# CELL: nasa_crosscheck
# Independent cross-check of the irradiance level: NASA POWER daily all-sky GHI (kWh/m²/day) for the same point.
NASA_RATIO = None
if not SMOKE_TEST:
    try:
        q = dict(parameters="ALLSKY_SFC_SW_DWN", community="RE", longitude=LON, latitude=LAT,
                 start=DATA_START.replace("-", ""), end=DATA_END.replace("-", ""), format="JSON")
        js = http_json("https://power.larc.nasa.gov/api/temporal/daily/point?" + urllib.parse.urlencode(q))
        s = pd.Series(js["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"], dtype=float).replace(-999.0, np.nan)
        nasa_daily = float(s.mean())
        om_daily = float(WX["ghi"].sum() / 1000 / (len(WX) / 24))
        NASA_RATIO = om_daily / nasa_daily
        print(f"Mean GHI: Open-Meteo ERA5 {om_daily:.2f} vs NASA POWER {nasa_daily:.2f} kWh/m²/day → ratio {NASA_RATIO:.3f}")
    except Exception as e:
        print("NASA POWER cross-check not available this run:", e)
else:
    print("Smoke test: NASA POWER cross-check skipped.")

## 3 · From sunlight to a notional 1 GW plant

A deliberately simple PV model, with every number a named assumption above:

`P = min(CAPACITY, CAPACITY × DC_AC_RATIO × GHI/1000 × PERF_RATIO × (1 + TEMP_COEFF × (T_cell − 25)))`,
with `T_cell = T_air + (NOCT − 20)/800 × GHI`.

It uses horizontal irradiance, so it ignores tilt and tracking. That shifts the level of output but hardly changes
how predictable tomorrow is, which is what this test measures.

In [ ]:
# CELL: pv_model
def pv_output_mw(ghi, t2m):
    t_cell = t2m + (NOCT_C - 20) / 800 * ghi
    p = CAPACITY_MW * DC_AC_RATIO * ghi / 1000 * PERF_RATIO * (1 + TEMP_COEFF * (t_cell - 25))
    return np.clip(p, 0, CAPACITY_MW)

WX["p_mw"] = pv_output_mw(WX["ghi"].values, WX["t2m"].values)
YEARS = sorted(WX.index.year.unique())
GHI_YEARLY = WX.groupby(WX.index.year)["ghi"].sum() / 1000 / WX.groupby(WX.index.year).size() * 24   # kWh/m²/day
CF_YEARLY = WX.groupby(WX.index.year)["p_mw"].mean() / CAPACITY_MW
DIURNAL = WX.groupby(WX.index.hour)["ghi"].mean()
PEAK_HOUR = int(DIURNAL.idxmax())
print("Mean GHI by year (kWh/m²/day):", GHI_YEARLY.round(2).to_dict())
print("Capacity factor by year:", (CF_YEARLY * 100).round(1).to_dict(), "%")
print("Sunniest hour label (IST, preceding-hour mean):", PEAK_HOUR)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw=dict(width_ratios=[1.5, 1]))
monthly = WX["p_mw"].resample("MS").sum() / 1000     # GWh per month
axes[0].bar(monthly.index, monthly.values, width=25, color=B.MOSS)
axes[0].set_ylabel("GWh per month")
B.chart_title(axes[0], "A notional 1 GW plant at Khavda", f"monthly energy · {YEARS[0]}-{YEARS[-1]} · ERA5 via Open-Meteo")
prof = WX.groupby([WX.index.month.isin([6, 7, 8, 9]), WX.index.hour])["p_mw"].mean().unstack(0)
axes[1].plot(prof.index, prof[False], color=B.MOSS, lw=2.4, label="Oct-May")
axes[1].plot(prof.index, prof[True], color=B.AMBER, lw=2.4, label="Monsoon (Jun-Sep)")
axes[1].set_xlabel("hour (IST)"); axes[1].set_ylabel("MW, average"); axes[1].legend()
B.chart_title(axes[1], "Average day", "dry season vs monsoon")
B.source_note(fig, f"23N Field Test #06 · {DATA_CREDIT} · Independent study on public data.")
plt.tight_layout(); plt.show()

## 4 · Method: three next-day forecasts for every day of the test year

For each day *D* the forecast is made from data up to the end of day *D−1* and covers the 24 hours of *D*. It
becomes that day's schedule.

1. **Same as yesterday (persistence):** the schedule for *D* is *D−1*'s output, hour by hour.
2. **7-day average:** the mean of the last seven days for each hour.
3. **Chronos-Bolt-Small:** `amazon/chronos-bolt-small` reads the last `CONTEXT_DAYS` days of hourly output and returns
   quantile forecasts for the next 24 hours; we use the median. It is pretrained on other time series and is
   **not trained on this data** (zero-shot).

All forecasts are clipped to 0…capacity, and set to zero at hours that were dark on all of the last seven days.

In [ ]:
# CELL: forecasters
P = WX["p_mw"].values.astype(np.float64)
DAYS = pd.DatetimeIndex(WX.index[::24].normalize())
assert len(P) == 24 * len(DAYS) and (WX.index[::24].hour == 0).all(), "hourly series must start at 00:00 and be complete"
PDAY = P.reshape(len(DAYS), 24)

def day_indices(year):
    return np.where(DAYS.year == year)[0]

def persistence(d):
    return PDAY[d - 1]

def seven_day(d):
    return PDAY[d - 7:d].mean(0)

def night_mask(d):
    return PDAY[d - 7:d].max(0) <= 0

if SMOKE_TEST:
    MODEL_LABEL = "STAND-IN (not Chronos)"
    HARDWARE = "CPU (smoke test)"
    def model_forecast(day_list):
        # SMOKE TEST ONLY: a blend of the two baselines, so the pipeline runs without downloading the model.
        return np.vstack([0.5 * persistence(d) + 0.5 * seven_day(d) for d in day_list])
else:
    import torch
    from chronos import BaseChronosPipeline
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    HARDWARE = torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU"
    PIPE = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEV, torch_dtype=torch.float32)
    def model_forecast(day_list):
        out = []
        for i in range(0, len(day_list), BATCH):
            ctx = [torch.tensor(P[(d - CONTEXT_DAYS) * 24:d * 24], dtype=torch.float32) for d in day_list[i:i + BATCH]]
            q, _ = PIPE.predict_quantiles(ctx, prediction_length=HORIZON_H, quantile_levels=[0.1, 0.5, 0.9])
            out.append(q[:, :, 1].float().cpu().numpy())
        return np.vstack(out)
print("Forecaster:", MODEL_LABEL, "| hardware:", HARDWARE)

def run_forecasts(year):
    ds = day_indices(year)
    assert ds.min() >= max(CONTEXT_DAYS, 7), "not enough history before the test year"
    t0 = time.time()
    fm = model_forecast(list(ds))
    secs = time.time() - t0
    rows = []
    for j, d in enumerate(ds):
        # no leakage: every input index is strictly before day d
        assert d - CONTEXT_DAYS >= 0 and d - 7 >= 0   # inputs are PDAY[:d] / P[:d*24] only (see the functions above)
        dark = night_mask(d)
        for name, f in (("persistence", persistence(d)), ("7-day average", seven_day(d)), ("model", fm[j])):
            f = np.clip(np.asarray(f, float), 0, CAPACITY_MW); f[dark] = 0.0
            for h in range(24):
                rows.append((DAYS[d] + pd.Timedelta(hours=h), name, PDAY[d, h], f[h], bool(dark[h])))
    fc = pd.DataFrame(rows, columns=["time", "method", "actual_mw", "sched_mw", "dark"])
    return fc, secs

FC, MODEL_SECS = run_forecasts(TEST_YEAR)
print(f"{TEST_YEAR}: {FC['time'].dt.normalize().nunique()} days forecast; model inference {MODEL_SECS:.1f}s")

## 5 · The deviation rule, as code

CERC DSM Regulations 2024, Regulation 8(4), for a solar seller, applied per 15-minute block:

- deviation % = |actual − schedule| ÷ (X % × available capacity + (100 − X) % × schedule). X is 100 % in
  FY 2026-27, so the denominator is available capacity (here 1,000 MW).
- up to `BAND1_PCT` (5 %): settled at the contract rate, so **no loss**;
- `BAND1_PCT`–`BAND2_PCT` (5–10 %): under-injection is paid back at 110 %, over-injection is received at 90 % → a
  **loss of 10 %** of the contract rate on that energy;
- beyond `BAND2_PCT`: under-injection is paid at 200 %, over-injection earns nothing → a **loss of 100 %**.

The charge below is that loss against being paid the contract rate for every kWh actually generated.

In [ ]:
# CELL: dsm
def deviation_loss_rs(actual, sched, band1=None, band2=None, x_pct=None):
    band1 = BAND1_PCT if band1 is None else band1
    band2 = BAND2_PCT if band2 is None else band2
    x_pct = X_PCT if x_pct is None else x_pct
    actual = np.asarray(actual, float); sched = np.asarray(sched, float)
    denom = x_pct / 100 * CAPACITY_MW + (100 - x_pct) / 100 * sched
    dev_mw = np.abs(actual - sched)
    safe = np.where(denom > 0, denom, 1.0)
    dev_pct = np.where(denom > 0, dev_mw / safe * 100, np.where(dev_mw > 0, 1e9, 0.0))
    mw_band2 = np.clip(np.minimum(dev_pct, band2) - band1, 0, None) / 100 * denom
    mw_beyond = np.clip(dev_pct - band2, 0, None) / 100 * denom
    mw_beyond = np.minimum(mw_beyond, dev_mw)                      # guard for a zero denominator
    hours = 1.0                                                     # each hourly row = 4 identical 15-min blocks
    kwh_band2 = mw_band2 * 1000 * hours; kwh_beyond = mw_beyond * 1000 * hours
    return CONTRACT_RATE_RS_PER_KWH * (BAND2_LOSS * kwh_band2 + BEYOND_LOSS * kwh_beyond)

def summarise(fc, **rule):
    out = {}
    for m, g in fc.groupby("method", sort=False):
        loss = deviation_loss_rs(g["actual_mw"].values, g["sched_mw"].values, **rule)
        day = ~g["dark"].values
        err = np.abs(g["actual_mw"].values - g["sched_mw"].values)
        out[m] = dict(charge_cr=float(loss.sum() / CRORE),
                      nmae_day_pct=float(err[day].mean() / CAPACITY_MW * 100),
                      rmse_day_pct=float(np.sqrt((err[day] ** 2).mean()) / CAPACITY_MW * 100),
                      within_band1_pct=float((err[day] / CAPACITY_MW * 100 <= BAND1_PCT).mean() * 100),
                      bias_mw=float((g["sched_mw"].values - g["actual_mw"].values)[day].mean()))
    return pd.DataFrame(out).T

FC["charge_rs"] = deviation_loss_rs(FC["actual_mw"].values, FC["sched_mw"].values)
SUM = summarise(FC)
ENERGY_GWH = float(FC[FC.method == "model"]["actual_mw"].sum() / 1000)
REVENUE_CR = ENERGY_GWH * 1e6 * CONTRACT_RATE_RS_PER_KWH / CRORE
SUM.round(2)

## 6 · Results: the headline, after the sanity checks

In [ ]:
# CELL: results
PEN_P, PEN_7, PEN_M = (SUM.loc[m, "charge_cr"] for m in ("persistence", "7-day average", "model"))
MAE_P, MAE_7, MAE_M = (SUM.loc[m, "nmae_day_pct"] for m in ("persistence", "7-day average", "model"))
SAVING_CR = PEN_P - PEN_M
CUT = SAVING_CR / PEN_P if PEN_P > 0 else float("nan")

# robustness variants
FC_R, _ = run_forecasts(ROBUST_YEAR)
SUM_R = summarise(FC_R)
SAVING_R = SUM_R.loc["persistence", "charge_cr"] - SUM_R.loc["model", "charge_cr"]
SUM_OLD = summarise(FC, band1=OLD_BAND1_PCT, band2=OLD_BAND2_PCT)
SUM_XN = summarise(FC, x_pct=X_NEXT_PCT)
mon = FC["time"].dt.month.isin([6, 7, 8, 9])
SUM_MONSOON, SUM_DRY = summarise(FC[mon]), summarise(FC[~mon])

# ---------------- sanity checks (before any headline is printed) ----------------
PROBLEMS, NOTES = [], []
for y in YEARS:
    n = int((WX.index.year == y).sum())
    if n not in (8760, 8784):
        PROBLEMS.append(f"{y} has {n} hourly rows (expected 8760/8784)")
if N_MISSING > 48:
    PROBLEMS.append(f"{N_MISSING} hours were missing and had to be filled")
g_mean = float(GHI_YEARLY.mean())
if not GHI_DAILY_RANGE[0] <= g_mean <= GHI_DAILY_RANGE[1]:
    PROBLEMS.append(f"mean GHI {g_mean:.2f} kWh/m²/day is outside {GHI_DAILY_RANGE}: units or location wrong?")
if WX["ghi"].max() > MAX_GHI:
    PROBLEMS.append(f"max GHI {WX['ghi'].max():.0f} W/m² > {MAX_GHI}: units wrong?")
if not PEAK_HOUR_RANGE[0] <= PEAK_HOUR <= PEAK_HOUR_RANGE[1]:
    PROBLEMS.append(f"sunniest hour label is {PEAK_HOUR}:00, outside {PEAK_HOUR_RANGE}: time zone wrong?")
cf = float(CF_YEARLY.mean())
if not CF_RANGE[0] <= cf <= CF_RANGE[1]:
    PROBLEMS.append(f"capacity factor {cf:.1%} outside {CF_RANGE}: PV model or units wrong?")
med = float(GHI_YEARLY.median())
for y, v in GHI_YEARLY.items():
    if abs(v / med - 1) * 100 > YEAR_BREAK_PCT:
        PROBLEMS.append(f"{y} mean GHI {v:.2f} differs from the median year {med:.2f} by > {YEAR_BREAK_PCT}%: data-version break?")
if NASA_RATIO is None:
    NOTES.append("NASA POWER cross-check unavailable this run (not blocking)")
elif not NASA_RATIO_RANGE[0] <= NASA_RATIO <= NASA_RATIO_RANGE[1]:
    PROBLEMS.append(f"Open-Meteo/NASA POWER GHI ratio {NASA_RATIO:.2f} outside {NASA_RATIO_RANGE}")
if not PERSIST_NMAE_RANGE[0] <= MAE_P <= PERSIST_NMAE_RANGE[1]:
    PROBLEMS.append(f"persistence daytime error {MAE_P:.1f}% of capacity outside {PERSIST_NMAE_RANGE}: bug?")
for m, v in SUM["charge_cr"].items():
    if not 0 <= v <= REVENUE_CR:
        PROBLEMS.append(f"{m} charge ₹{v:.1f} cr is outside 0..revenue (₹{REVENUE_CR:.0f} cr)")
if CUT == CUT and CUT > MAX_CREDIBLE_CUT:
    PROBLEMS.append(f"model cuts charges by {CUT:.0%}: implausibly good for zero-shot, check for leakage")
if MAE_M > 1.5 * MAE_7:
    PROBLEMS.append(f"model error {MAE_M:.1f}% is far worse than the 7-day average ({MAE_7:.1f}%): model misused? (context, units)")
if np.sign(SAVING_CR) != np.sign(SAVING_R):
    NOTES.append(f"robustness: the saving changes sign between {TEST_YEAR} and {ROBUST_YEAR}: headline is NOT robust, say so")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE TEST: synthetic weather and a stand-in forecaster")

print("Assumptions:", dict(CAPACITY_MW=CAPACITY_MW, DC_AC_RATIO=DC_AC_RATIO, PERF_RATIO=PERF_RATIO,
      CONTRACT_RATE_RS_PER_KWH=CONTRACT_RATE_RS_PER_KWH, BAND1_PCT=BAND1_PCT, BAND2_PCT=BAND2_PCT, X_PCT=X_PCT,
      CONTEXT_DAYS=CONTEXT_DAYS, TEST_YEAR=TEST_YEAR, MODEL=MODEL_LABEL))
print(f"Plant: {ENERGY_GWH:,.0f} GWh in {TEST_YEAR} (CF {cf:.1%}), revenue at the contract rate ≈ ₹{REVENUE_CR:,.0f} cr")
for n in NOTES:
    print("note:", n)
if PROBLEMS:
    print("\n" + "!" * 78)
    for p in PROBLEMS:
        print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every failure above is explained or fixed.")
    print("!" * 78 + "\n")
else:
    print("All sanity checks passed.")

def fmt_cr(v):
    return f"{v:.1f}" if abs(v) < 100 else f"{v:,.0f}"

CHANGE = f"{abs(CUT) * 100:.0f}% {'lower' if SAVING_CR >= 0 else 'higher'}"
if SAVING_CR >= 0:
    HEADLINE = (f"a {MODEL_PARAMS}-parameter open model cut estimated deviation charges for a notional 1 GW plant at Khavda "
                f"from ₹{fmt_cr(PEN_P)} crore to ₹{fmt_cr(PEN_M)} crore a year, against \"same as yesterday\" ({TEST_YEAR}).")
    BIG = f"₹{fmt_cr(SAVING_CR)} cr"
    BIG_LINE = f"a year saved per GW vs \"same as yesterday\" ({CHANGE} deviation charges)"
else:
    HEADLINE = (f"a {MODEL_PARAMS}-parameter open model did NOT beat \"same as yesterday\": estimated deviation charges for a "
                f"notional 1 GW plant at Khavda rose from ₹{fmt_cr(PEN_P)} crore to ₹{fmt_cr(PEN_M)} crore a year ({TEST_YEAR}).")
    BIG = f"−₹{fmt_cr(-SAVING_CR)} cr"
    BIG_LINE = f"a year per GW: the model was worse than \"same as yesterday\""
print("\nHEADLINE (estimate):", HEADLINE)
print(f"Daytime forecast error (MAE, % of capacity): persistence {MAE_P:.1f} · 7-day average {MAE_7:.1f} · {MODEL_LABEL} {MAE_M:.1f}")
print(f"7-day average alone: ₹{fmt_cr(PEN_7)} cr/yr")
print("\nRobustness:")
print(f"  {ROBUST_YEAR} instead of {TEST_YEAR}: persistence ₹{SUM_R.loc['persistence','charge_cr']:.1f} cr → model ₹{SUM_R.loc['model','charge_cr']:.1f} cr (saving ₹{SAVING_R:.1f} cr)")
print(f"  old bands ±{OLD_BAND1_PCT}/{OLD_BAND2_PCT}% (until Mar 2026): persistence ₹{SUM_OLD.loc['persistence','charge_cr']:.1f} cr → model ₹{SUM_OLD.loc['model','charge_cr']:.1f} cr")
print(f"  X = {X_NEXT_PCT}% (FY 2027-28 denominator): persistence ₹{SUM_XN.loc['persistence','charge_cr']:.1f} cr → model ₹{SUM_XN.loc['model','charge_cr']:.1f} cr")
print(f"  monsoon only (Jun-Sep): ₹{SUM_MONSOON.loc['persistence','charge_cr']:.1f} → ₹{SUM_MONSOON.loc['model','charge_cr']:.1f} cr · rest of year: ₹{SUM_DRY.loc['persistence','charge_cr']:.1f} → ₹{SUM_DRY.loc['model','charge_cr']:.1f} cr")

In [ ]:
# CELL: results_chart
METHOD_ORDER = ["persistence", "7-day average", "model"]
METHOD_NAME = {"persistence": "Same as yesterday", "7-day average": "7-day average", "model": MODEL_LABEL}
METHOD_COLOR = {"persistence": B.AMBER, "7-day average": B.STONE, "model": B.MOSS}
MONTHLY = (FC.assign(month=FC["time"].dt.month).groupby(["month", "method"])["charge_rs"].sum().unstack()[METHOD_ORDER] / CRORE)

def draw_monthly(ax, grow=1.0, big=False):
    x = np.arange(12); w = 0.27
    for k, m in enumerate(METHOD_ORDER):
        ax.bar(x + (k - 1) * w, MONTHLY[m].reindex(range(1, 13)).fillna(0).values * grow, w,
               color=METHOD_COLOR[m], label=METHOD_NAME[m])
    ax.set_xticks(x, ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"], fontsize=15 if big else 10)
    ax.set_ylim(0, float(MONTHLY.values.max()) * 1.18 + 1e-9)
    ax.set_ylabel("₹ crore", fontsize=14 if big else 10); ax.tick_params(axis="y", labelsize=13 if big else 10)
    ax.legend(loc="upper left", fontsize=14 if big else 9, ncol=3 if big else 1)

fig, ax = plt.subplots(figsize=(10, 4.6))
draw_monthly(ax)
B.chart_title(ax, "Estimated deviation charges by month", f"notional 1 GW at Khavda · {TEST_YEAR} · CERC DSM 2024 Reg 8(4), ±{BAND1_PCT}% band")
B.source_note(fig, f"23N Field Test #06 · {DATA_CREDIT} · estimates · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft06_results.png", dpi=150); plt.show()

# the costliest week for "same as yesterday", shown honestly as such
daily_pen = FC[FC.method == "persistence"].groupby(FC["time"].dt.normalize())["charge_rs"].sum()
wk_start = daily_pen.rolling(7).sum().idxmax() - pd.Timedelta(days=6)
WEEK = FC[(FC["time"] >= wk_start) & (FC["time"] < wk_start + pd.Timedelta(days=7))]
fig, ax = plt.subplots(figsize=(12, 4))
a = WEEK[WEEK.method == "model"]
ax.plot(a["time"], a["actual_mw"], color=B.INK, lw=2.2, label="Actual (modelled from ERA5)")
for m in ("persistence", "model"):
    g = WEEK[WEEK.method == m]
    ax.plot(g["time"], g["sched_mw"], color=METHOD_COLOR[m], lw=1.8, label=METHOD_NAME[m], ls="--" if m == "persistence" else "-")
ax.set_ylabel("MW"); ax.legend(ncol=3, loc="upper left")
B.chart_title(ax, "The costliest week for \"same as yesterday\"", f"{wk_start:%d %b %Y} + 6 days · forecast made at midnight for the next day")
plt.tight_layout(); fig.savefig("ft06_week.png", dpi=150); plt.show()

## 7 · Interactive demo
Pick a day of the test year: you get the actual output, the three schedules, the ±5 % and ±10 % bands and the
estimated charge for that day. The cell doesn't block "Run all"; open the public `gradio.live` link it prints.

In [ ]:
# CELL: gradio
DAY_CHOICES = [d.strftime("%Y-%m-%d") for d in sorted(FC["time"].dt.normalize().unique())]

def day_figure(day):
    d = pd.Timestamp(day)
    g = FC[FC["time"].dt.normalize() == d]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    a = g[g.method == "model"]
    hrs = a["time"].dt.hour.values
    ax.plot(hrs, a["actual_mw"], color=B.INK, lw=2.4, label="Actual")
    txt = []
    for m in METHOD_ORDER:
        s = g[g.method == m]
        ax.plot(hrs, s["sched_mw"], color=METHOD_COLOR[m], lw=1.8, label=METHOD_NAME[m])
        txt.append(f"{METHOD_NAME[m]}: ₹{s['charge_rs'].sum() / 1e5:.1f} lakh")
    ax.set_xlabel("hour (IST)"); ax.set_ylabel("MW"); ax.legend(fontsize=9)
    B.chart_title(ax, f"{d:%d %b %Y}", " · ".join(txt))
    plt.tight_layout()
    return fig

try:
    if SMOKE_TEST:
        raise RuntimeError("skipped in smoke test")
    import gradio as gr
    with gr.Blocks(title="23N Field Test #06 · Tomorrow's sun at Khavda") as demo:
        gr.Markdown(f"## Tomorrow's sun at Khavda\nNext-day forecasts for a notional 1 GW plant, {TEST_YEAR}. "
                    "Estimates on ERA5 reanalysis. Independent study on public data by 23N AI Labs.")
        dd = gr.Dropdown(DAY_CHOICES, value=DAY_CHOICES[len(DAY_CHOICES) // 2], label="Day")
        out = gr.Plot()
        dd.change(day_figure, dd, out)
        demo.load(day_figure, dd, out)
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", e)

## 8 · Video (1080×1080, 30 fps, 58 s, H.264) and cover (1080×1350)
Every curve and number in the video comes from the run above. The week shown is labelled for what it is: the
costliest week of the year for "same as yesterday".

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30

def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def build_video_ctx():
    a = WEEK[WEEK.method == "model"]
    hrs = np.arange(len(a))
    sched = {m: WEEK[WEEK.method == m]["sched_mw"].values for m in METHOD_ORDER}
    # the single costliest day for persistence inside that week, for the band illustration
    wk_day = WEEK[WEEK.method == "persistence"].groupby(WEEK["time"].dt.normalize())["charge_rs"].sum().idxmax()
    gd = FC[FC["time"].dt.normalize() == wk_day]
    return dict(hrs=hrs, actual=a["actual_mw"].values, sched=sched, wk_start=wk_start,
                day=wk_day, day_actual=gd[gd.method == "model"]["actual_mw"].values,
                day_p=gd[gd.method == "persistence"]["sched_mw"].values,
                day_m=gd[gd.method == "model"]["sched_mw"].values,
                day_charge_p=gd[gd.method == "persistence"]["charge_rs"].sum(),
                day_charge_m=gd[gd.method == "model"]["charge_rs"].sum())

def _title(fig, title, sub):
    fig.text(0.055, 0.85, title, fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
    fig.text(0.055, 0.785, sub.upper(), fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")

def draw_evidence(fig, t, c):
    """t runs 0..35 inside the evidence section (video seconds 5..40)."""
    B.header(fig, FT_NUM)
    if t < 12:                                                     # one week: actual vs two schedules
        _title(fig, "One week at Khavda", f"costliest week for 'same as yesterday' · {c['wk_start']:%d %b %Y}")
        ax = fig.add_axes([0.10, 0.20, 0.85, 0.53])
        n = len(c["hrs"]); k = max(2, int(n * _ease(_prog(t, 0.3, 4.5))))
        ax.plot(c["hrs"][:k], c["actual"][:k], color=B.INK, lw=3, label="Actual")
        if t > 4.8:
            k2 = max(2, int(n * _ease(_prog(t, 4.8, 7.5))))
            ax.plot(c["hrs"][:k2], c["sched"]["persistence"][:k2], color=B.AMBER, lw=2.4, ls="--", label="Same as yesterday")
        if t > 7.8:
            k3 = max(2, int(n * _ease(_prog(t, 7.8, 10.5))))
            ax.plot(c["hrs"][:k3], c["sched"]["model"][:k3], color=B.MOSS, lw=2.6, label=MODEL_LABEL)
        ax.set_xlim(0, n); ax.set_ylim(0, CAPACITY_MW * 1.32)
        ax.set_xticks(np.arange(0, n, 24), [f"day {i + 1}" for i in range(n // 24)], fontsize=13)
        ax.set_yticks(np.arange(0, CAPACITY_MW + 1, 200))
        ax.tick_params(axis="y", labelsize=13); ax.set_ylabel("MW", fontsize=14)
        ax.legend(loc="upper right", fontsize=14, ncol=3)
        B.caption(fig, "Each schedule is fixed at midnight for the next 24 hours")
    elif t < 23:                                                   # the rule on one day
        _title(fig, "The rule: ±5% is free", f"{c['day']:%d %b %Y} · bands around the schedule")
        ax = fig.add_axes([0.10, 0.22, 0.85, 0.51])
        h = np.arange(24); use_m = t >= 17.5
        s = c["day_m"] if use_m else c["day_p"]; col = B.MOSS if use_m else B.AMBER
        b1, b2 = BAND1_PCT / 100 * CAPACITY_MW, BAND2_PCT / 100 * CAPACITY_MW
        al = _ease(_prog(t, 12.2, 13.5))
        ax.fill_between(h, s - b2, s + b2, color=B.PAPER_DEEP, alpha=al, lw=0, label=f"±{BAND2_PCT}%: 10% charge")
        ax.fill_between(h, s - b1, s + b1, color=B.MOSS_LIGHT, alpha=0.55 * al, lw=0, label=f"±{BAND1_PCT}%: free")
        ax.plot(h, s, color=col, lw=2.6, label="Same as yesterday" if not use_m else MODEL_LABEL)
        ax.plot(h, c["day_actual"], color=B.INK, lw=3, label="Actual")
        ax.set_xlim(0, 23); ax.set_ylim(0, CAPACITY_MW * 1.12); ax.set_xlabel("hour (IST)", fontsize=13)
        ax.tick_params(labelsize=13); ax.set_ylabel("MW", fontsize=14); ax.legend(loc="upper left", fontsize=13)
        ch = c["day_charge_m"] if use_m else c["day_charge_p"]
        fig.text(0.945, 0.785, f"₹{ch / 1e5:,.0f} LAKH THAT DAY", fontproperties=B.FP.get("mono_bold"), fontsize=16,
                 color=col, ha="right", va="top")
        B.caption(fig, "Beyond ±10% of capacity the energy is effectively lost: paid back at 200% or earning 0")
    else:                                                          # the year
        _title(fig, f"All of {TEST_YEAR}", f"estimated charges per month · notional 1 GW · ₹{CONTRACT_RATE_RS_PER_KWH}/kWh")
        ax = fig.add_axes([0.11, 0.22, 0.84, 0.50])
        draw_monthly(ax, grow=_ease(_prog(t, 23.2, 27)), big=True)
        if t > 28:
            fig.text(0.945, 0.785, f"₹{fmt_cr(PEN_P)} CR → ₹{fmt_cr(PEN_M)} CR", fontproperties=B.FP.get("mono_bold"),
                     fontsize=16, color=B.MOSS, ha="right", va="top", alpha=_prog(t, 28, 28.6))
        B.caption(fig, f"Daytime error: {MAE_P:.1f}% of capacity for 'same as yesterday', {MAE_M:.1f}% for the model")

def render_video(c, path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "Can a small AI forecast tomorrow's sun at Khavda?",
                f"{len(YEARS)} years of ERA5 · {MODEL_PARAMS} open model · CERC deviation rules"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        if t < 12:   key = ("w", min(round(t * 15) / 15, 10.6))
        elif t < 23: key = ("d", min(round(t * 15) / 15, 13.6) if t < 17.5 else "m")
        else:        key = ("y", min(round(t * 15) / 15, 28.7))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t, c); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, BIG, BIG_LINE,
                               f"{MODEL_LABEL} zero-shot, next-day forecasts for {TEST_YEAR}, CERC DSM 2024 Reg 8(4), ERA5 reanalysis. Estimate.")
    v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    print("video frames:", v.frames, f"({v.frames / FPS:.1f}s)")

def render_cover(c, path=COVER_PATH):
    def panel(ax):
        ax.set_axis_on(); draw_monthly(ax, big=True)
        pos = ax.get_position()                        # leave room for the y tick labels inside the page margin
        ax.set_position([pos.x0 + 0.06, pos.y0 + 0.02, pos.width - 0.06, pos.height - 0.02])
        ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
        ax.set_ylabel("₹ crore per month", fontsize=13)
    B.cover(path, FT_NUM, "Tomorrow's sun at Khavda", f"next-day solar forecasts · {TEST_YEAR} · notional 1 GW",
            BIG, BIG_LINE, image=panel, data_credit=DATA_CREDIT)

In [ ]:
# CELL: video_render
VCTX = build_video_ctx()
render_video(VCTX)
render_cover(VCTX)
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 9 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """Can a small open AI model forecast tomorrow's solar output at Khavda better than "same as yesterday"? And what is that worth?

23N Field Test #06: {headline}

Since 1 April 2026, CERC's deviation rules for solar sellers charge for any miss beyond ±5% of capacity in a 15-minute block, down from ±10%.

How we tested it:
• {n_years} years of hourly ERA5 sunlight for the Khavda RE park, Kutch (via Open-Meteo), as output of a notional 1 GW plant
• {n_days} next-day forecasts for {test_year}: "same as yesterday", a 7-day average, and {model_name} ({model_params} parameters, zero-shot, free Colab)
• Charges under CERC DSM Regulations 2024, Reg 8(4), at an assumed ₹{rate}/kWh contract rate

Result: estimated deviation charges ₹{pen_persist} cr → ₹{pen_model} cr a year per GW ({change}). Daytime error {mae_persist}% → {mae_model}% of capacity.

Limits: reanalysis weather, not meter data; hourly, not 15-minute; no intraday revisions. An estimate, not a plant's bill.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """Can a {model_params}-parameter open AI model forecast tomorrow's Khavda solar output better than "same as yesterday"? Est. deviation charges per GW, {test_year}: ₹{pen_persist} cr → ₹{pen_model} cr ({change}). 23N Field Test #06. Notebook below.

---
Thread (optional):
1/ Data: {n_years} years of hourly ERA5 sunlight (Open-Meteo, CC BY 4.0) for the Khavda RE park, turned into a notional 1 GW plant's output.
2/ Rule: CERC DSM 2024 Reg 8(4) for solar, ±{band1}% free band, then 10% and 100% of the contract rate. Daytime error {mae_persist}% → {mae_model}% of capacity.
3/ Limits: reanalysis, not meter data; hourly, not 15-min; no intraday revisions. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
POST_VALUES = dict(headline=HEADLINE[0].upper() + HEADLINE[1:], n_years=len(YEARS), n_days=FC["time"].dt.normalize().nunique(),
                   test_year=TEST_YEAR, model_name=MODEL_LABEL, model_params=MODEL_PARAMS,
                   rate=f"{CONTRACT_RATE_RS_PER_KWH:.2f}", pen_persist=fmt_cr(PEN_P), pen_model=fmt_cr(PEN_M),
                   change=CHANGE, mae_persist=f"{MAE_P:.1f}", mae_model=f"{MAE_M:.1f}", band1=BAND1_PCT,
                   signature=B.post_signature(FT_NUM) + " #SolarEnergy #Forecasting #OpenSourceAI")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 10 · Limitations (read before quoting the number)

- **Reanalysis, not meter data.** "Actual" output is modelled from ERA5 reanalysis at a ~25 km grid cell, not
  measured at any plant. ERA5 smooths clouds, so real plants are harder to forecast and real charges are likely
  higher for every method. The comparison between methods is the finding, not the rupee level.
- **Hourly, not 15-minute.** DSM settles 15-minute blocks; each hour here counts as four identical blocks, which
  hides intra-hour swings.
- **No intraday revisions.** Sellers may revise schedules during the day, and good operators do. This models a
  seller that fixes the schedule at midnight and never revises it, so it is an upper bound on charges.
- **Simple PV model.** Horizontal irradiance, a single performance ratio, no tilt, tracking, outages or
  curtailment. Available capacity is taken as 1,000 MW in every block.
- **Rule and price are assumptions.** The bands follow CERC DSM Regulations 2024, Reg 8(4) for solar sellers with
  X = 100 % (FY 2026-27). The regulations are under challenge in the Delhi High Court. The contract rate is one
  auction's price; plants have different PPAs. Change the named variables and re-run.
- **One site, one model, zero-shot.** Chronos-Bolt-Small is not tuned on this data. A model given a weather
  forecast (numerical weather prediction) as input would very likely do better; this test asks what a
  history-only small model can do.
- **Not a statement about any real plant, developer or operator.** Khavda is used only as a location.
- **Licences.** ERA5 via Open-Meteo: CC BY 4.0 ("Weather data by Open-Meteo.com"; contains modified Copernicus
  Climate Change Service information). NASA POWER: NASA open data. Chronos-Bolt-Small: Apache 2.0.